In [1]:
%pip install numpy pandas

import numpy as np
import pandas as pd
from pathlib import Path

Note: you may need to restart the kernel to use updated packages.


In [2]:
BASE_PATH = Path(r"D:/DATASETS/cnn-lstm")

INPUT_BINARY = BASE_PATH / "dataset_binary.bin"
INPUT_INDEX = BASE_PATH / "dataset_index.csv"

OUTPUT_BINARY = BASE_PATH / "processed_dataset_binary.bin"
OUTPUT_INDEX = BASE_PATH / "processed_dataset_index.csv"

SAMPLE_RATE = 128                                       # Hz
WINDOW_SECONDS = 2                                      # window duration
WINDOW_OVERLAP = 0.5                                    # 50% overlap
SAMPLES_PER_WINDOW = int(SAMPLE_RATE * WINDOW_SECONDS)
STEP = int(SAMPLES_PER_WINDOW * (1 - WINDOW_OVERLAP))   # step size for sliding window
FREQUENCIES = np.arange(2.0, 40.5, 0.5)                 # target frequency bins

ELECTRODE_CHANNELS = ["Fp1","Fp2","F3","F4","C3",
                      "C4","P3","P4","O1","O2",
                      "F7","F8","T7","T8","P7",
                      "P8","Fz","Cz","Pz"]

In [3]:
# Load the index and memory-map the binary (lazy loading — no full read into RAM)
index_df = pd.read_csv(INPUT_INDEX)
data = np.memmap(INPUT_BINARY, dtype='float64', mode='r')

print(f"Index entries : {len(index_df)}")
print(f"Binary values : {len(data):,} float64")
print(f"Labels        : {index_df['label'].unique()}")
print(f"Columns       : {index_df.columns.tolist()}")

Index entries : 561
Binary values : 399,216,163 float64
Labels        : ['Control' 'ADHD-C' 'ADHD-I' 'ADHD-H']
Columns       : ['path', 'start_idx', 'length', 'rows', 'cols', 'label']


In [10]:
%pip install mne scikit-learn scipy

import mne

# Use a fixed component count: leave 1 degree of freedom (n_channels - 1)
ICA_N_COMPONENTS = len(["Fp1","Fp2","F3","F4","C3",
                         "C4","P3","P4","O1","O2",
                         "F7","F8","T7","T8","P7",
                         "P8","Fz","Cz","Pz"]) - 1  # = 18

def apply_ica_cleaning(df: pd.DataFrame, sfreq: float):
    """Apply ICA to remove muscle and eye artifacts from EEG dataframe."""

    ch_names = df.columns.tolist()
    ch_types = ["eeg"] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(df.to_numpy().T, info, verbose=False)

    # --- Filter for ICA (1–80 Hz typical) ---
    raw.filter(1.0, 60.0, fir_design="firwin", verbose=False)

    # --- Fit ICA ---
    ica = mne.preprocessing.ICA(n_components=ICA_N_COMPONENTS,
                                 method="fastica", verbose=False)
    ica.fit(raw, picks="eeg", decim=3)

    # --- Detect EOG-like (blink) components if present ---
    try:
        eog_inds, _ = ica.find_bads_eog(raw)
    except Exception:
        eog_inds = []

    # --- Detect muscle-like components (high-frequency ratio heuristic) ---
    sources = ica.get_sources(raw).get_data()
    n_ic = sources.shape[0]
    sfreq = raw.info["sfreq"]

    from scipy.signal import welch
    muscle_candidates = []
    for ic_idx in range(n_ic):
        f, Pxx = welch(sources[ic_idx], sfreq, nperseg=512)
        hf_mask = (f >= 20)
        hf_ratio = Pxx[hf_mask].sum() / np.sum(Pxx)
        if hf_ratio > 0.25:  # threshold – tune as needed
            muscle_candidates.append(ic_idx)

    exclude = list(set(eog_inds + muscle_candidates))
    ica.exclude = exclude
    if len(exclude) > 0:
        print(f"ICA removed components: {exclude}")
        raw_clean = ica.apply(raw.copy())
    else:
        print("No ICA components removed.")
        raw_clean = raw

    # --- Convert back to pandas DataFrame ---
    cleaned = pd.DataFrame(raw_clean.get_data().T, columns=ch_names)
    return cleaned


Note: you may need to restart the kernel to use updated packages.


In [5]:
def apply_filter(df: pd.DataFrame, lower=0.5, upper=30):
    electrode_columns = df.select_dtypes(include=[np.number]).columns

    n = len(df)
    freqs = np.fft.rfftfreq(n, d=1 / SAMPLE_RATE)
    filtered_df = df.copy()

    for electrode in electrode_columns:
        signal = df[electrode].to_numpy()
        fft_vals = np.fft.rfft(signal)
        fft_filter = (freqs >= lower) & (freqs <= upper)
        fft_vals[~fft_filter] = 0
        filtered_signal = np.fft.irfft(fft_vals, n=n)
        filtered_df[electrode] = filtered_signal

    return filtered_df

In [6]:
def process_window(window: pd.DataFrame) -> np.ndarray:
    """Convert a time-domain EEG window to frequency-domain power.

    Returns
    -------
    np.ndarray of shape (n_freqs, n_channels)
        Power spectrum interpolated to FREQUENCIES for each electrode.
    """
    n = len(window)
    original_freqs = np.fft.rfftfreq(n, d=1 / SAMPLE_RATE)

    power_matrix = np.zeros((len(FREQUENCIES), len(ELECTRODE_CHANNELS)),
                            dtype=np.float64)

    for ch_idx, electrode in enumerate(ELECTRODE_CHANNELS):
        sig = window[electrode].to_numpy()
        fft_vals = np.fft.rfft(sig)
        power = np.abs(fft_vals) ** 2
        power_matrix[:, ch_idx] = np.interp(FREQUENCIES, original_freqs, power)

    return power_matrix

In [7]:
def apply_ica_cleaning_to_dataset(df: pd.DataFrame):
    result_df = df.copy()
    numerical_df = df.select_dtypes(include=[np.number])
    cleaned_df = apply_ica_cleaning(numerical_df, SAMPLE_RATE)
    for col in numerical_df.columns:
        result_df[col] = cleaned_df[col]

    return cleaned_df

In [8]:
from scipy import signal
from scipy.integrate import simpson

# Define frequency bands (Hz)
bands = {
    'Theta':      (4,  8),
    'Alpha':      (8,  13),
    'Fast_Alpha': (8,  10),
    'Beta':       (13, 30),
    'High_Beta':  (15, 30),
}

# Broadband range used as the denominator for relative power
BROADBAND = (0.5, 40.0)


def compute_relative_band_powers(channel_data: np.ndarray, fs: float) -> dict:
    """Compute relative band power for all bands for a single channel.

    Relative power = band_power / total_broadband_power
    where total_broadband_power is integrated over BROADBAND.
    A single Welch PSD is computed and reused for all bands.
    """
    freqs, psd = signal.welch(channel_data, fs,
                              nperseg=min(256, len(channel_data)))

    # Total broadband power (denominator)
    bb_mask    = (freqs >= BROADBAND[0]) & (freqs <= BROADBAND[1])
    total_power = simpson(psd[bb_mask], freqs[bb_mask])

    rel_powers = {}
    for band_name, (low, high) in bands.items():
        idx = (freqs >= low) & (freqs <= high)
        abs_power = simpson(psd[idx], freqs[idx])
        rel_powers[band_name] = (abs_power / total_power
                                 if total_power > 0 else np.nan)

    return rel_powers

In [11]:
import gc

output_records = []   # rows for the output index CSV
output_offset  = 0    # running byte-offset into the output binary

with open(OUTPUT_BINARY, 'wb') as f_out:
    for i, row in index_df.iterrows():
        subject_id = Path(row['path']).stem
        label      = row['label']
        print(f"\n[{i+1}/{len(index_df)}] {subject_id}  ({label})")

        # ── 1. Lazy-load subject from binary ──────────────────────────
        chunk = data[row['start_idx'] : row['start_idx'] + row['length']]
        subject_df = pd.DataFrame(
            chunk.reshape(row['rows'], row['cols']),
            columns=ELECTRODE_CHANNELS,
        )

        # ── 2. Preprocessing pipeline ─────────────────────────────────
        subject_df = apply_filter(subject_df, 0.5, 30)          # bandpass
        subject_df = apply_ica_cleaning_to_dataset(subject_df)   # ICA artefact removal

        # ── 3. Per-subject relative band-power features (before z-norm) ──
        # Accumulate relative powers across channels, then average
        rel_acc = {b: 0.0 for b in bands}
        for ch in ELECTRODE_CHANNELS:
            ch_rel = compute_relative_band_powers(subject_df[ch].values, SAMPLE_RATE)
            for bname in bands:
                rel_acc[bname] += ch_rel[bname]
        for bname in rel_acc:
            rel_acc[bname] /= len(ELECTRODE_CHANNELS)

        # Relative-power ratios
        tbr = rel_acc['Theta'] / rel_acc['Beta']  if rel_acc['Beta']  else np.nan
        tar = rel_acc['Theta'] / rel_acc['Alpha'] if rel_acc['Alpha'] else np.nan

        # ── 4. Subject-wise z-score normalisation ─────────────────────
        subject_df = (subject_df - subject_df.mean()) / subject_df.std()

        # ── 5. Sliding-window FFT → write to binary ──────────────────
        n_samples  = len(subject_df)
        window_num = 0

        for start in range(0, n_samples - SAMPLES_PER_WINDOW + 1, STEP):
            window = subject_df.iloc[start : start + SAMPLES_PER_WINDOW]
            power_matrix = process_window(window)          # (n_freqs, n_channels)

            flat = power_matrix.ravel()                    # row-major
            flat.tofile(f_out)

            output_records.append({
                'id':              subject_id,
                'label':           label,
                'window':          window_num,
                'start_idx':       output_offset,
                'length':          flat.size,
                'rows':            power_matrix.shape[0],  # len(FREQUENCIES)
                'cols':            power_matrix.shape[1],  # len(ELECTRODE_CHANNELS)
                'rel_theta':       rel_acc['Theta'],
                'rel_alpha':       rel_acc['Alpha'],
                'rel_fast_alpha':  rel_acc['Fast_Alpha'],
                'rel_beta':        rel_acc['Beta'],
                'rel_high_beta':   rel_acc['High_Beta'],
                'tbr':             tbr,
                'tar':             tar,
            })
            output_offset += flat.size
            window_num    += 1

        print(f"  → {window_num} windows")
        del chunk, subject_df
        gc.collect()

print(f"\n{'='*50}")
print(f"Total windows written : {len(output_records)}")
print(f"Output binary values  : {output_offset:,}")


[1/561] v107  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 153 windows

[2/561] v108  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 147 windows

[3/561] v109  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
ICA removed components: [8, 10]
Applying ICA to Raw instance
    Transforming to ICA space (18 components)
    Zeroing out 2 ICA components
    Projecting back using 19 PCA components
  → 124 windows

[4/561] v10p  (ADHD-C)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.0s.
No ICA components removed.
  → 110 windows

[5/561] v110  (

c:\Users\michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\decomposition\_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 93 windows

[15/561] v121  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.5s.
No ICA components removed.
  → 126 windows

[16/561] v123  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 112 windows

[17/561] v125  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 119 windows

[18/561] v127  (Control)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.4s.
ICA removed components: [16, 11, 4, 7]
Applying ICA to Raw instance
    Transforming to ICA space (18 component

c:\Users\michael\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\decomposition\_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


No ICA components removed.
  → 156 windows

[46/561] v209  (ADHD-I)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.1s.
No ICA components removed.
  → 235 windows

[47/561] v20p  (ADHD-C)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.2s.
No ICA components removed.
  → 275 windows

[48/561] v213  (ADHD-H)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.0s.
No ICA components removed.
  → 95 windows

[49/561] v215  (ADHD-H)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.2s.
ICA removed components: [14]
Applying ICA to Raw instance
    Transforming to ICA space (18 components)
    Zeroing out 1 ICA component
    Projecting back using 19 PCA comp

...

  → 467 windows

[560/561] sub-88075681_ses-1_restEO  (ADHD-I)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.3s.
ICA removed components: [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 16]
Applying ICA to Raw instance
    Transforming to ICA space (18 components)
    Zeroing out 15 ICA components
    Projecting back using 19 PCA components


C:\Users\michael\AppData\Local\Temp\ipykernel_33684\2162253564.py:25: RuntimeWarning: Using n_components=18 (resulting in n_components_=18) may lead to an unstable mixing matrix estimation because the ratio between the largest (16) and smallest (1.1e-31) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 3
  ica.fit(raw, picks="eeg", decim=3)


  → 467 windows

[561/561] sub-88076173_ses-1_restEO  (ADHD-I)
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 18 components
Fitting ICA took 0.2s.
ICA removed components: [4, 5, 8, 13, 14, 15, 16]
Applying ICA to Raw instance
    Transforming to ICA space (18 components)
    Zeroing out 7 ICA components
    Projecting back using 19 PCA components


C:\Users\michael\AppData\Local\Temp\ipykernel_33684\2162253564.py:25: RuntimeWarning: Using n_components=18 (resulting in n_components_=18) may lead to an unstable mixing matrix estimation because the ratio between the largest (7.7) and smallest (1.6e-31) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 5
  ica.fit(raw, picks="eeg", decim=3)


  → 467 windows

Total windows written : 163428
Output binary values  : 239,095,164


In [12]:
# Save the output index CSV
output_index_df = pd.DataFrame(output_records)
output_index_df.to_csv(OUTPUT_INDEX, index=False)

print(f"Saved binary → {OUTPUT_BINARY}")
print(f"Saved index  → {OUTPUT_INDEX}")
print(f"\nIndex shape: {output_index_df.shape}")
print(f"\nLabel distribution:")
print(output_index_df['label'].value_counts().to_string())
display(output_index_df.head(10))

Saved binary → D:\DATASETS\cnn-lstm\processed_dataset_binary.bin
Saved index  → D:\DATASETS\cnn-lstm\processed_dataset_index.csv

Index shape: (163428, 14)

Label distribution:
label
ADHD-I     50270
Control    48522
ADHD-H     40810
ADHD-C     23826


,id,label,window,start_idx,length,rows,cols,rel_theta,rel_alpha,rel_fast_alpha,rel_beta,rel_high_beta,tbr,tar
0,v107,Control,0,0,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
1,v107,Control,1,1463,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
2,v107,Control,2,2926,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
3,v107,Control,3,4389,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
4,v107,Control,4,5852,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
5,v107,Control,5,7315,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
6,v107,Control,6,8778,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
7,v107,Control,7,10241,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
8,v107,Control,8,11704,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008
9,v107,Control,9,13167,1463,77,19,0.107009,0.034079,0.021371,0.02265,0.016063,4.724506,3.140008


In [14]:
# Verify: read back the first window from the output binary
verify = np.memmap(OUTPUT_BINARY, dtype='float64', mode='r')
sample = output_index_df.iloc[0]
window_arr = verify[sample['start_idx'] : sample['start_idx'] + sample['length']]\
    .reshape(sample['rows'], sample['cols'])

print(f"Verified window shape: {window_arr.shape}  "
      f"(expected {len(FREQUENCIES)} freqs × {len(ELECTRODE_CHANNELS)} channels)")
print(f"\nFirst 5 freqs × first 5 channels:")
display(pd.DataFrame(
    window_arr[:5, :5],
    index=FREQUENCIES[:5],
    columns=ELECTRODE_CHANNELS[:5],
))

Verified window shape: (77, 19)  (expected 77 freqs × 19 channels)

First 5 freqs × first 5 channels:


,Fp1,Fp2,F3,F4,C3
2.0,267.000197,576.459938,1126.595173,178.910625,134.144174
2.5,1039.491622,1490.554961,1311.906474,554.888633,3200.998414
3.0,510.536204,831.366992,1408.830356,47.281497,584.700500
3.5,748.725232,159.473546,1424.573468,629.678740,1239.338875
4.0,71.641515,637.717579,722.079258,274.251011,1489.789000
